In [1]:
import sys
import os
import pysrt
import pygame
import html
from typing import Optional, List
from PySide6.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
    QPushButton, QLabel, QFileDialog, QStyle, QSlider, QMessageBox,
    QStatusBar, QMenuBar
)
from PySide6.QtCore import Qt, QTimer
from PySide6.QtGui import QFont, QIcon, QAction, QActionGroup

# --- Constants ---
UPDATE_INTERVAL_MS = 50
DEFAULT_FONT_SIZE = 16
SUBTITLE_FONT_SIZE = 100  # Increased dramatically!
INITIAL_VOLUME = 70

class SrtAudioPlayer(QMainWindow):
    THEME_DARK = "dark"
    THEME_LIGHT = "light"

    DARK_THEME_COLORS = {
        "app_bg": "#333333", "app_fg": "#E0E0E0",
        "btn_bg": "#555555", "btn_border": "#666666",
        "btn_hover": "#686868", "btn_pressed": "#4A4A4A",
        "btn_disabled_bg": "#404040", "btn_disabled_fg": "#888888",
        "label_fg": "#D0D0D0",
        "slider_groove_border": "#4A4A4A", "slider_groove_bg": "#404040",
        "slider_handle_bg": "#77AADD", "slider_handle_border": "#5588CC",
        "slider_sub_page_bg": "#6699CC",
        "statusbar_bg": "#282828", "statusbar_fg": "#E0E0E0",
        "menu_bar_bg": "#333333", "menu_bar_fg": "#E0E0E0",
        "menu_bg": "#383838", "menu_fg": "#E0E0E0", "menu_border": "#4A4A4A", "menu_item_selected_bg": "#5A5A5A",
        "subtitle_bg": "#282828", "subtitle_fg": "#E0E0E0", "subtitle_border": "#444444",
        "subtitle_highlight_word": "#FFD700"  # Yellow
    }

    LIGHT_THEME_COLORS = {
        "app_bg": "#F0F0F0", "app_fg": "#101010",
        "btn_bg": "#DDDDDD", "btn_border": "#B0B0B0",
        "btn_hover": "#C8C8C8", "btn_pressed": "#B0B0B0",
        "btn_disabled_bg": "#E0E0E0", "btn_disabled_fg": "#777777",
        "label_fg": "#202020",
        "slider_groove_border": "#C0C0C0", "slider_groove_bg": "#D0D0D0",
        "slider_handle_bg": "#4A86C3", "slider_handle_border": "#386FA0",
        "slider_sub_page_bg": "#6699CC",
        "statusbar_bg": "#E0E0E0", "statusbar_fg": "#101010",
        "menu_bar_bg": "#F0F0F0", "menu_bar_fg": "#101010",
        "menu_bg": "#F5F5F5", "menu_fg": "#101010", "menu_border": "#C0C0C0", "menu_item_selected_bg": "#D5D5D5",
        "subtitle_bg": "#FFFFFF", "subtitle_fg": "#000000", "subtitle_border": "#BBBBBB",
        "subtitle_highlight_word": "#D2691E"  # Chocolate / Dark Orange
    }

    def __init__(self):
        super().__init__()
        self.setWindowTitle("SRT Audio Synchronizer Pro")
        self.setGeometry(100, 100, 1000, 700)  # Bigger window
        self.current_theme = self.THEME_DARK
        self.theme_colors = self.DARK_THEME_COLORS
        self.srt_path: Optional[str] = None
        self.audio_path: Optional[str] = None
        self.subs: Optional[List[pysrt.srtitem.SubRipItem]] = None
        self.current_sub_index: int = -1
        self.audio_duration_ms: int = 0
        self.is_paused: bool = False
        self.user_is_seeking: bool = False
        self.playback_start_offset_ms: int = 0

        try:
            pygame.mixer.init()
            pygame.mixer.music.set_volume(INITIAL_VOLUME / 100.0)
        except pygame.error as e:
            QMessageBox.critical(self, "Pygame Error", f"Error initializing pygame.mixer: {e}\nAudio playback will be disabled.")
            self.pygame_mixer_failed = True
        else:
            self.pygame_mixer_failed = False

        self._create_menus()

        self.central_widget = QWidget()
        self.setCentralWidget(self.central_widget)
        main_layout = QVBoxLayout(self.central_widget)

        file_area_layout = QHBoxLayout()
        self.btn_load_srt = QPushButton("Load SRT")
        self.lbl_srt_file = QLabel("SRT: Not loaded")
        self.btn_load_audio = QPushButton("Load Audio")
        self.lbl_audio_file = QLabel("Audio: Not loaded")
        file_area_layout.addWidget(self.btn_load_srt)
        file_area_layout.addWidget(self.lbl_srt_file, 1)
        file_area_layout.addWidget(self.btn_load_audio)
        file_area_layout.addWidget(self.lbl_audio_file, 1)
        main_layout.addLayout(file_area_layout)

        self.lbl_subtitle = QLabel("Load an SRT and Audio file to begin.")
        self.lbl_subtitle.setObjectName("SubtitleLabel")
        self.lbl_subtitle.setAlignment(Qt.AlignCenter)
        font_subtitle = QFont()
        font_subtitle.setPointSize(SUBTITLE_FONT_SIZE)
        font_subtitle.setBold(True)
        self.lbl_subtitle.setFont(font_subtitle)
        self.lbl_subtitle.setWordWrap(True)
        self.lbl_subtitle.setMinimumHeight(400)  # Larger area
        main_layout.addWidget(self.lbl_subtitle)

        timeline_layout = QHBoxLayout()
        self.lbl_current_time = QLabel("00:00:00")
        self.progress_slider = QSlider(Qt.Horizontal)
        self.progress_slider.setEnabled(False)
        self.progress_slider.setToolTip("Seek audio position")
        self.lbl_total_time = QLabel("00:00:00")
        timeline_layout.addWidget(self.lbl_current_time)
        timeline_layout.addWidget(self.progress_slider, 1)
        timeline_layout.addWidget(self.lbl_total_time)
        main_layout.addLayout(timeline_layout)

        control_layout = QHBoxLayout()
        self.btn_play_pause = QPushButton()
        self.btn_play_pause.setToolTip("Play/Pause (Spacebar)")
        self.btn_play_pause.setFixedSize(50, 50)
        play_pause_action = QAction("Play/Pause", self)
        play_pause_action.setShortcut(Qt.Key_Space)
        play_pause_action.triggered.connect(self.toggle_play_pause)
        self.addAction(play_pause_action)

        self.btn_stop = QPushButton()
        self.btn_stop.setToolTip("Stop (Ctrl+S)")
        self.btn_stop.setFixedSize(50, 50)
        stop_action = QAction("Stop", self)
        stop_action.setShortcut(Qt.CTRL | Qt.Key_S)
        stop_action.triggered.connect(self.stop_playback)
        self.addAction(stop_action)

        self.volume_slider = QSlider(Qt.Horizontal)
        self.volume_slider.setRange(0, 100)
        self.volume_slider.setValue(INITIAL_VOLUME)
        self.volume_slider.setFixedWidth(150)
        self.volume_slider.setToolTip("Volume")

        self.lbl_volume_icon = QLabel()

        control_layout.addStretch()
        control_layout.addWidget(self.btn_play_pause)
        control_layout.addWidget(self.btn_stop)
        control_layout.addStretch()
        control_layout.addWidget(self.lbl_volume_icon)
        control_layout.addWidget(self.volume_slider)
        main_layout.addLayout(control_layout)

        self.status_bar = QStatusBar()
        self.setStatusBar(self.status_bar)
        self.status_bar.showMessage("Ready.")

        self.btn_load_srt.clicked.connect(self.load_srt)
        self.btn_load_audio.clicked.connect(self.load_audio)
        self.btn_play_pause.clicked.connect(self.toggle_play_pause)
        self.btn_stop.clicked.connect(self.stop_playback)
        self.progress_slider.sliderMoved.connect(self.seek_audio_slider_moved)
        self.progress_slider.sliderPressed.connect(self.slider_pressed)
        self.progress_slider.sliderReleased.connect(self.slider_released)
        self.volume_slider.valueChanged.connect(self.set_volume)

        self.timer = QTimer(self)
        self.timer.setInterval(UPDATE_INTERVAL_MS)
        self.timer.timeout.connect(self.update_display)

        self.apply_stylesheet()
        self.update_icons()
        self.update_button_states()

        if self.pygame_mixer_failed:
            self.disable_audio_controls()

    def _create_menus(self):
        menu_bar = self.menuBar()
        theme_menu = menu_bar.addMenu("&Theme")
        self.dark_theme_action = QAction("Dark Mode", self, checkable=True)
        self.dark_theme_action.triggered.connect(lambda: self.set_theme(self.THEME_DARK))
        theme_menu.addAction(self.dark_theme_action)
        self.light_theme_action = QAction("Light Mode", self, checkable=True)
        self.light_theme_action.triggered.connect(lambda: self.set_theme(self.THEME_LIGHT))
        theme_menu.addAction(self.light_theme_action)
        theme_group = QActionGroup(self)
        theme_group.setExclusive(True)
        theme_group.addAction(self.dark_theme_action)
        theme_group.addAction(self.light_theme_action)
        if self.current_theme == self.THEME_DARK:
            self.dark_theme_action.setChecked(True)
        else:
            self.light_theme_action.setChecked(True)

    def set_theme(self, theme_name):
        if theme_name == self.THEME_DARK:
            self.current_theme = self.THEME_DARK
            self.theme_colors = self.DARK_THEME_COLORS
            self.dark_theme_action.setChecked(True)
        elif theme_name == self.THEME_LIGHT:
            self.current_theme = self.THEME_LIGHT
            self.theme_colors = self.LIGHT_THEME_COLORS
            self.light_theme_action.setChecked(True)
        else:
            return
        self.apply_stylesheet()
        font_subtitle = QFont()
        font_subtitle.setPointSize(SUBTITLE_FONT_SIZE)
        font_subtitle.setBold(True)
        self.lbl_subtitle.setFont(font_subtitle)
        self.update_icons()
        current_pos_ms = self.progress_slider.value() if self.audio_duration_ms > 0 else 0
        self.update_display(forced_time_ms=current_pos_ms)

    def update_icons(self):
        self.setWindowIcon(QIcon(self.style().standardIcon(QStyle.SP_MediaPlay)))
        self.btn_load_srt.setIcon(QIcon(self.style().standardIcon(QStyle.SP_FileIcon)))
        self.btn_load_audio.setIcon(QIcon(self.style().standardIcon(QStyle.SP_MediaVolume)))
        self.btn_stop.setIcon(QIcon(self.style().standardIcon(QStyle.SP_MediaStop)))
        self.lbl_volume_icon.setPixmap(self.style().standardIcon(QStyle.SP_MediaVolume).pixmap(24, 24))
        self.update_button_states()

    def disable_audio_controls(self):
        self.btn_load_audio.setEnabled(False)
        self.btn_play_pause.setEnabled(False)
        self.btn_stop.setEnabled(False)
        self.progress_slider.setEnabled(False)
        self.volume_slider.setEnabled(False)
        self.lbl_audio_file.setText("Audio: Unavailable (Mixer Error)")
        self.status_bar.showMessage("Audio system error. Playback disabled.")

    def apply_stylesheet(self):
        colors = self.theme_colors
        self.setStyleSheet(f"""
            QMainWindow, QWidget {{
                background-color: {colors['app_bg']}; 
                color: {colors['app_fg']}; 
                font-size: {DEFAULT_FONT_SIZE}px;
            }}
            QPushButton {{
                background-color: {colors['btn_bg']}; 
                border: 1px solid {colors['btn_border']}; 
                padding: 8px 15px;
                border-radius: 4px; 
                min-width: 80px;
                color: {colors['app_fg']}; 
            }}
            QPushButton:hover {{ background-color: {colors['btn_hover']}; }}
            QPushButton:pressed {{ background-color: {colors['btn_pressed']}; }}
            QPushButton:disabled {{ 
                background-color: {colors['btn_disabled_bg']}; 
                color: {colors['btn_disabled_fg']}; 
                border-color: {colors['btn_disabled_bg']};
            }}
            QLabel {{ color: {colors['label_fg']}; }}
            QLabel#SubtitleLabel {{
                padding: 20px;
                border: 2px solid {colors['subtitle_border']};
                background-color: {colors['subtitle_bg']};
                color: {colors['subtitle_fg']}; 
                border-radius: 8px;
            }}
            QSlider::groove:horizontal {{
                border: 1px solid {colors['slider_groove_border']}; 
                height: 8px; 
                background: {colors['slider_groove_bg']};
                margin: 2px 0; 
                border-radius: 4px;
            }}
            QSlider::handle:horizontal {{
                background: {colors['slider_handle_bg']}; 
                border: 1px solid {colors['slider_handle_border']}; 
                width: 16px;
                margin: -4px 0; 
                border-radius: 8px;
            }}
            QSlider::sub-page:horizontal {{ 
                background: {colors['slider_sub_page_bg']}; 
                border-radius: 4px; 
            }}
            QStatusBar {{ 
                background-color: {colors['statusbar_bg']}; 
                color: {colors['statusbar_fg']};
            }}
            QMenuBar {{ 
                background-color: {colors['menu_bar_bg']}; 
                color: {colors['menu_bar_fg']}; 
            }}
            QMenuBar::item {{ 
                background-color: transparent; 
                padding: 4px 10px; 
                margin: 2px;
                border-radius: 3px;
            }}
            QMenuBar::item:selected {{ 
                background-color: {colors['menu_item_selected_bg']}; 
            }}
            QMenu {{ 
                background-color: {colors['menu_bg']}; 
                color: {colors['menu_fg']}; 
                border: 1px solid {colors['menu_border']}; 
                padding: 2px;
            }}
            QMenu::item {{ padding: 5px 20px 5px 20px; }}
            QMenu::item:selected {{ 
                background-color: {colors['menu_item_selected_bg']}; 
            }}
            QMenu::separator {{ height: 1px; background-color: {colors['menu_border']}; margin: 4px 0; }}
        """)

    def load_srt(self):
        path, _ = QFileDialog.getOpenFileName(self, "Open SRT File", "", "SubRip Text Files (*.srt)")
        if path:
            self.srt_path = path
            try:
                self.subs = pysrt.open(self.srt_path, encoding='utf-8-sig')
                for sub in self.subs:
                    sub.start_ms = sub.start.ordinal
                    sub.end_ms = sub.end.ordinal
                self.lbl_srt_file.setText(f"SRT: {os.path.basename(path)}")
                self.status_bar.showMessage(f"Loaded {len(self.subs)} subtitles from {os.path.basename(path)}.", 5000)
                if not self.audio_path:
                    self.lbl_subtitle.setText("SRT loaded. Now load an audio file.")
            except Exception as e:
                self.lbl_srt_file.setText("SRT: Error loading!")
                QMessageBox.warning(self, "SRT Load Error", f"Could not load SRT file: {e}")
                self.subs = None
                self.srt_path = None
            self.reset_playback_state()
            self.update_button_states()

    def load_audio(self):
        if self.pygame_mixer_failed:
            return
        path, _ = QFileDialog.getOpenFileName(self, "Open Audio File", "", "Audio Files (*.mp3 *.wav *.ogg)")
        if path:
            self.audio_path = path
            try:
                if pygame.mixer.music.get_busy():
                    pygame.mixer.music.stop()
                pygame.mixer.music.load(self.audio_path)
                sound = pygame.mixer.Sound(self.audio_path)
                self.audio_duration_ms = int(sound.get_length() * 1000)
                del sound
                self.lbl_audio_file.setText(f"Audio: {os.path.basename(path)}")
                self.lbl_total_time.setText(self.format_time(self.audio_duration_ms))
                self.progress_slider.setMaximum(self.audio_duration_ms)
                self.status_bar.showMessage(f"Loaded audio: {os.path.basename(path)}.", 5000)
                if not self.subs:
                    self.lbl_subtitle.setText("Audio loaded. Now load an SRT file.")
                elif self.subs and self.audio_path:
                    self.lbl_subtitle.setText("Ready to play. Press Play or Spacebar.")
            except pygame.error as e:
                self.lbl_audio_file.setText("Audio: Error loading!")
                QMessageBox.warning(self, "Audio Load Error", f"Could not load audio file: {e}")
                self.audio_path = None
                self.audio_duration_ms = 0
            self.reset_playback_state()
            self.update_button_states()

    def update_button_states(self):
        if self.pygame_mixer_failed:
            self.disable_audio_controls()
            return
        can_operate = bool(self.srt_path and self.audio_path and self.subs and self.audio_duration_ms > 0)
        self.btn_play_pause.setEnabled(can_operate)
        self.progress_slider.setEnabled(can_operate)
        is_playing_or_paused = pygame.mixer.music.get_busy() or self.is_paused
        self.btn_stop.setEnabled(can_operate and is_playing_or_paused)
        if pygame.mixer.music.get_busy() and not self.is_paused:
            self.btn_play_pause.setIcon(self.style().standardIcon(QStyle.SP_MediaPause))
            self.btn_play_pause.setToolTip("Pause (Spacebar)")
        else:
            self.btn_play_pause.setIcon(self.style().standardIcon(QStyle.SP_MediaPlay))
            self.btn_play_pause.setToolTip("Play (Spacebar)")

    def toggle_play_pause(self):
        if self.pygame_mixer_failed or not self.audio_path or not self.subs:
            return
        if pygame.mixer.music.get_busy() and not self.is_paused:
            pygame.mixer.music.pause()
            self.is_paused = True
            self.timer.stop()
            self.status_bar.showMessage("Paused.", 3000)
        else:
            if self.is_paused:
                pygame.mixer.music.unpause()
                self.status_bar.showMessage("Resumed.", 3000)
            else:
                self.playback_start_offset_ms = self.progress_slider.value()
                try:
                    pygame.mixer.music.play(loops=0, start=self.playback_start_offset_ms / 1000.0)
                except pygame.error as e:
                    QMessageBox.warning(self, "Playback Error", f"Could not start playback: {e}")
                    self.reset_playback_state()
                    return
                self.status_bar.showMessage("Playing...", 3000)
            self.is_paused = False
            self.timer.start()
        self.update_button_states()

    def stop_playback(self):
        if self.pygame_mixer_failed:
            return
        if pygame.mixer.music.get_busy() or self.is_paused:
            pygame.mixer.music.stop()
            self.timer.stop()
            self.lbl_subtitle.setText("...")
            self.current_sub_index = -1
            self.is_paused = False
            self.progress_slider.setValue(0)
            self.lbl_current_time.setText(self.format_time(0))
            self.playback_start_offset_ms = 0
            self.status_bar.showMessage("Stopped.", 3000)
        self.update_button_states()

    def reset_playback_state(self):
        if pygame.mixer.music.get_busy() or self.is_paused:
            self.stop_playback()
        else:
            self.timer.stop()
            self.progress_slider.setValue(0)
            self.lbl_current_time.setText(self.format_time(0))
            self.playback_start_offset_ms = 0
        self.current_sub_index = -1
        if self.subs and self.audio_path:
            self.lbl_subtitle.setText("Ready to play. Press Play or Spacebar.")
        elif self.subs:
            self.lbl_subtitle.setText("SRT loaded. Load audio.")
        elif self.audio_path:
            self.lbl_subtitle.setText("Audio loaded. Load SRT.")
        else:
            self.lbl_subtitle.setText("Load an SRT and Audio file to begin.")
        if self.audio_duration_ms > 0:
            self.lbl_total_time.setText(self.format_time(self.audio_duration_ms))
        else:
            self.lbl_total_time.setText("00:00:00")
            self.progress_slider.setMaximum(0)
        self.update_button_states()

    def format_time(self, ms: int) -> str:
        s = max(0, ms // 1000)
        m, s = divmod(s, 60)
        h, m = divmod(m, 60)
        return f"{h:02d}:{m:02d}:{s:02d}"

    def update_display(self, forced_time_ms: Optional[int] = None):
        if self.pygame_mixer_failed:
            return
        actual_current_song_time_ms: int
        if forced_time_ms is not None:
            actual_current_song_time_ms = forced_time_ms
        elif pygame.mixer.music.get_busy() or self.is_paused:
            time_elapsed_since_play_call = pygame.mixer.music.get_pos()
            if time_elapsed_since_play_call == -1:
                if not self.is_paused and self.timer.isActive():
                    if self.audio_duration_ms > 0 and self.playback_start_offset_ms + pygame.mixer.music.get_pos() >= self.audio_duration_ms - (2 * UPDATE_INTERVAL_MS):
                        self.stop_playback()
                        self.status_bar.showMessage("Playback finished.", 3000)
                    else:
                        self.stop_playback()
                return
            actual_current_song_time_ms = self.playback_start_offset_ms + time_elapsed_since_play_call
            if not self.user_is_seeking and pygame.mixer.music.get_busy() and not self.is_paused:
                if self.audio_duration_ms > 0:
                    display_slider_time = min(actual_current_song_time_ms, self.audio_duration_ms)
                    self.progress_slider.setValue(display_slider_time)
        else:
            if self.timer.isActive():
                self.stop_playback()
                self.status_bar.showMessage("Playback finished.", 3000)
            return
        if self.audio_duration_ms > 0:
            actual_current_song_time_ms = min(actual_current_song_time_ms, self.audio_duration_ms)
        actual_current_song_time_ms = max(0, actual_current_song_time_ms)
        self.lbl_current_time.setText(self.format_time(actual_current_song_time_ms))
        if not self.subs:
            return
        active_sub = None
        for i, sub in enumerate(self.subs):
            if sub.start_ms <= actual_current_song_time_ms < sub.end_ms:
                active_sub = sub
                if self.current_sub_index != i:
                    self.current_sub_index = i
                break
        if active_sub:
            words = active_sub.text_without_tags.split()
            num_words = len(words)
            if num_words == 0:
                self.lbl_subtitle.setText(html.escape(active_sub.text_without_tags))
                return
            time_into_sub = actual_current_song_time_ms - active_sub.start_ms
            sub_duration = active_sub.end_ms - active_sub.start_ms
            highlight_idx = -1
            if sub_duration > 0:
                progress_ratio = time_into_sub / sub_duration
                highlight_idx = int(progress_ratio * num_words)
                highlight_idx = min(highlight_idx, num_words - 1)
            display_text_parts = []
            highlight_color = self.theme_colors['subtitle_highlight_word']
            font_size_px = SUBTITLE_FONT_SIZE
            for i, word in enumerate(words):
                escaped_word = html.escape(word)
                if i == highlight_idx:
                    display_text_parts.append(
                        f"<span style='font-size: {font_size_px}px; color: {highlight_color}; font-weight: bold;'>{escaped_word}</span>"
                    )
                else:
                    display_text_parts.append(
                        f"<span style='font-size: {font_size_px}px;'>{escaped_word}</span>"
                    )
            self.lbl_subtitle.setText(" ".join(display_text_parts))
        elif self.current_sub_index != -1:
            self.lbl_subtitle.setText("")
            self.current_sub_index = -1
        self.update_button_states()

    def slider_pressed(self):
        if self.pygame_mixer_failed:
            return
        self.user_is_seeking = True
        if pygame.mixer.music.get_busy() and not self.is_paused:
            pygame.mixer.music.pause()

    def slider_released(self):
        if self.pygame_mixer_failed:
            return
        self.user_is_seeking = False
        if not self.audio_path:
            return
        seek_ms = self.progress_slider.value()
        self.playback_start_offset_ms = seek_ms
        try:
            pygame.mixer.music.play(loops=0, start=seek_ms / 1000.0)
            if self.is_paused:
                pygame.mixer.music.pause()
            else:
                self.timer.start()
            self.status_bar.showMessage(f"Seeked to {self.format_time(seek_ms)}", 3000)
        except pygame.error as e:
            QMessageBox.warning(self, "Seek Error", f"Could not seek audio: {e}")
            self.reset_playback_state()
        self.update_display(forced_time_ms=seek_ms)
        self.update_button_states()

    def seek_audio_slider_moved(self, value_ms: int):
        if self.pygame_mixer_failed:
            return
        self.lbl_current_time.setText(self.format_time(value_ms))

    def set_volume(self, value: int):
        if self.pygame_mixer_failed:
            return
        pygame.mixer.music.set_volume(value / 100.0)
        self.status_bar.showMessage(f"Volume: {value}%", 2000)

    def closeEvent(self, event):
        self.stop_playback()
        if not self.pygame_mixer_failed:
            pygame.mixer.music.stop()
            pygame.mixer.quit()
        print("Application closed.")
        event.accept()


if __name__ == "__main__":
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)
    if sys.platform == "win32":
        import ctypes
        myappid = u'mycompany.myproduct.srtaudiosyncpro.1.4'
        try:
            ctypes.windll.shell32.SetCurrentProcessExplicitAppUserModelID(myappid)
        except AttributeError:
            print("Warning: Could not set AppUserModelID (usually for Windows taskbar icon).")
    window = SrtAudioPlayer()
    window.show()
    sys.exit(app.exec())

pygame 2.6.1 (SDL 2.28.4, Python 3.9.21)
Hello from the pygame community. https://www.pygame.org/contribute.html
Application closed.


SystemExit: 0

c:\Users\ANAS\anaconda3\envs\ML\lib\site-packages\IPython\core\interactiveshell.py:3513: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
import sys
import os
import pysrt # For SubRipItem, SubRipTime
import pygame
import html
from typing import Optional, List, Any
from PySide6.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
    QPushButton, QLabel, QFileDialog, QStyle, QSlider, QMessageBox,
    QStatusBar, QMenuBar, QTextEdit
)
from PySide6.QtCore import Qt, QTimer, QEvent
from PySide6.QtGui import QFont, QIcon, QAction, QActionGroup

# --- Constants ---
UPDATE_INTERVAL_MS = 50
DEFAULT_FONT_SIZE = 16
SUBTITLE_DISPLAY_FONT_SIZE = 70 # Slightly reduced for balance
TEXT_INPUT_FONT_SIZE = 22       # Slightly increased for prominence
INITIAL_VOLUME = 70

class TextAudioSynchronizerPro(QMainWindow): # Renamed class
    THEME_DARK = "dark"
    THEME_LIGHT = "light"

    DARK_THEME_COLORS = {
        "app_bg": "#333333", "app_fg": "#E0E0E0",
        "btn_bg": "#555555", "btn_border": "#666666",
        "btn_hover": "#686868", "btn_pressed": "#4A4A4A",
        "btn_disabled_bg": "#404040", "btn_disabled_fg": "#888888",
        "label_fg": "#D0D0D0", "label_info_fg": "#A0A0A0",
        "slider_groove_border": "#4A4A4A", "slider_groove_bg": "#404040",
        "slider_handle_bg": "#77AADD", "slider_handle_border": "#5588CC",
        "slider_sub_page_bg": "#6699CC",
        "statusbar_bg": "#282828", "statusbar_fg": "#E0E0E0",
        "menu_bar_bg": "#333333", "menu_bar_fg": "#E0E0E0",
        "menu_bg": "#383838", "menu_fg": "#E0E0E0", "menu_border": "#4A4A4A", "menu_item_selected_bg": "#5A5A5A",
        "subtitle_display_bg": "#2C2C2C", "subtitle_display_fg": "#E0E0E0", "subtitle_display_border": "#4A4A4A",
        "subtitle_highlight_word": "#FFD700",  # Yellow
        "text_edit_bg": "#3A3A3A", "text_edit_fg": "#E0E0E0", "text_edit_border": "#555555", "text_edit_placeholder": "#777777",
    }

    LIGHT_THEME_COLORS = {
        "app_bg": "#F0F0F0", "app_fg": "#101010",
        "btn_bg": "#DDDDDD", "btn_border": "#B0B0B0",
        "btn_hover": "#C8C8C8", "btn_pressed": "#B0B0B0",
        "btn_disabled_bg": "#E0E0E0", "btn_disabled_fg": "#777777",
        "label_fg": "#202020", "label_info_fg": "#505050",
        "slider_groove_border": "#C0C0C0", "slider_groove_bg": "#D0D0D0",
        "slider_handle_bg": "#4A86C3", "slider_handle_border": "#386FA0",
        "slider_sub_page_bg": "#6699CC",
        "statusbar_bg": "#E0E0E0", "statusbar_fg": "#101010",
        "menu_bar_bg": "#F0F0F0", "menu_bar_fg": "#101010",
        "menu_bg": "#F5F5F5", "menu_fg": "#101010", "menu_border": "#C0C0C0", "menu_item_selected_bg": "#D5D5D5",
        "subtitle_display_bg": "#FAFAFA", "subtitle_display_fg": "#000000", "subtitle_display_border": "#BDBDBD",
        "subtitle_highlight_word": "#D2691E",  # Chocolate / Dark Orange
        "text_edit_bg": "#FFFFFF", "text_edit_fg": "#000000", "text_edit_border": "#A0A0A0", "text_edit_placeholder": "#888888",
    }

    def __init__(self):
        super().__init__()
        self.setWindowTitle("Text-Audio Synchronizer Pro")
        self.setGeometry(100, 100, 1000, 750)
        self.current_theme = self.THEME_DARK
        self.theme_colors = self.DARK_THEME_COLORS

        self.audio_path: Optional[str] = None
        self.subs: Optional[List[pysrt.srtitem.SubRipItem]] = None
        self.current_sub_index: int = -1
        self.audio_duration_ms: int = 0
        self.is_paused: bool = False
        self.user_is_seeking: bool = False
        self.playback_start_offset_ms: int = 0
        self.current_typed_text: str = ""

        try:
            pygame.mixer.init()
            pygame.mixer.music.set_volume(INITIAL_VOLUME / 100.0)
        except pygame.error as e:
            QMessageBox.critical(self, "Pygame Error", f"Error initializing pygame.mixer: {e}\nAudio playback will be disabled.")
            self.pygame_mixer_failed = True
        else:
            self.pygame_mixer_failed = False

        self._create_menus()

        self.central_widget = QWidget()
        self.setCentralWidget(self.central_widget)
        main_layout = QVBoxLayout(self.central_widget)
        main_layout.setSpacing(15) # Add some general spacing
        main_layout.setContentsMargins(15, 15, 15, 15)


        # --- Text Input Area ---
        self.text_input_area = QTextEdit()
        self.text_input_area.setObjectName("TextInputArea")
        self.text_input_area.setPlaceholderText("1. Type your text here.\n2. Press Enter to choose an audio file.")
        font_input = QFont()
        font_input.setPointSize(TEXT_INPUT_FONT_SIZE)
        self.text_input_area.setFont(font_input)
        self.text_input_area.setFixedHeight(120)
        self.text_input_area.installEventFilter(self)
        main_layout.addWidget(self.text_input_area)

        # --- Audio File Info Label ---
        self.lbl_audio_file = QLabel("Audio: Not loaded. Type text and press Enter to begin.")
        self.lbl_audio_file.setObjectName("AudioInfoLabel")
        self.lbl_audio_file.setAlignment(Qt.AlignCenter)
        font_info = QFont()
        font_info.setPointSize(DEFAULT_FONT_SIZE - 2) # Slightly smaller for info
        self.lbl_audio_file.setFont(font_info)
        main_layout.addWidget(self.lbl_audio_file)
        main_layout.addSpacing(5)


        # --- Subtitle Display Label (for highlighting) ---
        self.subtitle_display_label = QLabel("Your synchronized text will appear here.")
        self.subtitle_display_label.setObjectName("SubtitleDisplayLabel")
        self.subtitle_display_label.setAlignment(Qt.AlignCenter)
        font_subtitle_display = QFont()
        font_subtitle_display.setPointSize(SUBTITLE_DISPLAY_FONT_SIZE)
        font_subtitle_display.setBold(True)
        self.subtitle_display_label.setFont(font_subtitle_display)
        self.subtitle_display_label.setWordWrap(True)
        self.subtitle_display_label.setMinimumHeight(280)
        main_layout.addWidget(self.subtitle_display_label, 1) # Allow it to stretch

        # --- Timeline ---
        timeline_layout = QHBoxLayout()
        self.lbl_current_time = QLabel("00:00:00")
        self.progress_slider = QSlider(Qt.Horizontal)
        self.progress_slider.setEnabled(False)
        self.progress_slider.setToolTip("Seek audio position")
        self.lbl_total_time = QLabel("00:00:00")
        timeline_layout.addWidget(self.lbl_current_time)
        timeline_layout.addWidget(self.progress_slider, 1)
        timeline_layout.addWidget(self.lbl_total_time)
        main_layout.addLayout(timeline_layout)

        # --- Controls ---
        control_layout = QHBoxLayout()
        self.btn_play_pause = QPushButton()
        self.btn_play_pause.setToolTip("Play/Pause (Spacebar)")
        self.btn_play_pause.setFixedSize(50, 50)
        play_pause_action = QAction("Play/Pause", self)
        play_pause_action.setShortcut(Qt.Key_Space)
        play_pause_action.triggered.connect(self.toggle_play_pause)
        self.addAction(play_pause_action)

        self.btn_stop = QPushButton()
        self.btn_stop.setToolTip("Stop (Ctrl+S)")
        self.btn_stop.setFixedSize(50, 50)
        stop_action = QAction("Stop", self)
        stop_action.setShortcut(Qt.CTRL | Qt.Key_S)
        stop_action.triggered.connect(self.stop_playback)
        self.addAction(stop_action)

        self.volume_slider = QSlider(Qt.Horizontal)
        self.volume_slider.setRange(0, 100)
        self.volume_slider.setValue(INITIAL_VOLUME)
        self.volume_slider.setFixedWidth(150)
        self.volume_slider.setToolTip("Volume")
        self.lbl_volume_icon = QLabel()

        control_layout.addStretch()
        control_layout.addWidget(self.btn_play_pause)
        control_layout.addWidget(self.btn_stop)
        control_layout.addStretch()
        control_layout.addWidget(self.lbl_volume_icon)
        control_layout.addWidget(self.volume_slider)
        main_layout.addLayout(control_layout)

        self.status_bar = QStatusBar()
        self.setStatusBar(self.status_bar)
        self.status_bar.showMessage("Ready. Type text and press Enter.")

        # Connections (NO btn_load_audio)
        self.btn_play_pause.clicked.connect(self.toggle_play_pause)
        self.btn_stop.clicked.connect(self.stop_playback)
        self.progress_slider.sliderMoved.connect(self.seek_audio_slider_moved)
        self.progress_slider.sliderPressed.connect(self.slider_pressed)
        self.progress_slider.sliderReleased.connect(self.slider_released)
        self.volume_slider.valueChanged.connect(self.set_volume)

        self.timer = QTimer(self)
        self.timer.setInterval(UPDATE_INTERVAL_MS)
        self.timer.timeout.connect(self.update_display)

        self.apply_stylesheet() # Apply initial theme
        self.update_icons()
        self.update_button_states()

        if self.pygame_mixer_failed:
            self.disable_audio_controls()
        else:
            self.text_input_area.setFocus() # Focus on text input on start

    def eventFilter(self, watched: QWidget, event: QEvent) -> bool:
        if watched == self.text_input_area and event.type() == QEvent.KeyPress:
            key_event = event
            if key_event.key() in (Qt.Key_Return, Qt.Key_Enter):
                if not (key_event.modifiers() & Qt.ShiftModifier):
                    self.handle_text_entry_and_trigger_audio_load()
                    return True # Event handled, consume it
        return super().eventFilter(watched, event)

    def handle_text_entry_and_trigger_audio_load(self):
        text = self.text_input_area.toPlainText().strip()
        if not text:
            QMessageBox.information(self, "No Text", "Please type some text before pressing Enter.")
            self.text_input_area.setFocus()
            return

        self.current_typed_text = text
        self.subs = None # Clear previous subs if any
        self.audio_path = None # Reset audio path as we are re-initiating
        self.audio_duration_ms = 0

        # Update UI to reflect text entry before audio dialog
        self.subtitle_display_label.setText(html.escape(text) + "\n\n(Opening audio selection...)")
        self.lbl_audio_file.setText("Audio: Awaiting selection...")
        self.lbl_total_time.setText(self.format_time(0))
        self.progress_slider.setMaximum(100) # Reset slider
        self.progress_slider.setValue(0)
        self.update_button_states() # Disable playback controls

        self.status_bar.showMessage("Text captured. Select an audio file.", 5000)
        QApplication.processEvents() # Ensure UI updates before dialog
        self.load_audio_dialog()


    def _create_menus(self):
        menu_bar = self.menuBar()
        # File Menu (Optional - could add "Reset" or "Exit")
        file_menu = menu_bar.addMenu("&File")
        reset_action = QAction(QIcon.fromTheme("document-revert", self.style().standardIcon(QStyle.SP_DialogResetButton)), "Reset All", self)
        reset_action.setShortcut(Qt.CTRL | Qt.Key_R)
        reset_action.triggered.connect(lambda: self.reset_playback_state(reset_text_input=True, from_menu=True))
        file_menu.addAction(reset_action)
        file_menu.addSeparator()
        exit_action = QAction(QIcon.fromTheme("application-exit", self.style().standardIcon(QStyle.SP_DialogCloseButton)), "E&xit", self)
        exit_action.setShortcut(Qt.CTRL | Qt.Key_Q)
        exit_action.triggered.connect(self.close)
        file_menu.addAction(exit_action)

        theme_menu = menu_bar.addMenu("&Theme")
        self.dark_theme_action = QAction("Dark Mode", self, checkable=True)
        self.dark_theme_action.triggered.connect(lambda: self.set_theme(self.THEME_DARK))
        theme_menu.addAction(self.dark_theme_action)
        self.light_theme_action = QAction("Light Mode", self, checkable=True)
        self.light_theme_action.triggered.connect(lambda: self.set_theme(self.THEME_LIGHT))
        theme_menu.addAction(self.light_theme_action)
        theme_group = QActionGroup(self)
        theme_group.setExclusive(True)
        theme_group.addAction(self.dark_theme_action)
        theme_group.addAction(self.light_theme_action)
        if self.current_theme == self.THEME_DARK:
            self.dark_theme_action.setChecked(True)
        else:
            self.light_theme_action.setChecked(True)

    def set_theme(self, theme_name):
        if theme_name == self.THEME_DARK:
            self.current_theme = self.THEME_DARK
            self.theme_colors = self.DARK_THEME_COLORS
            if hasattr(self, 'dark_theme_action'): self.dark_theme_action.setChecked(True)
        elif theme_name == self.THEME_LIGHT:
            self.current_theme = self.THEME_LIGHT
            self.theme_colors = self.LIGHT_THEME_COLORS
            if hasattr(self, 'light_theme_action'): self.light_theme_action.setChecked(True)
        else:
            return

        self.apply_stylesheet() # This will re-apply all styles

        # Re-apply fonts as stylesheet might override them if not specific enough
        font_subtitle_display = QFont()
        font_subtitle_display.setPointSize(SUBTITLE_DISPLAY_FONT_SIZE)
        font_subtitle_display.setBold(True)
        self.subtitle_display_label.setFont(font_subtitle_display)

        font_input = QFont()
        font_input.setPointSize(TEXT_INPUT_FONT_SIZE)
        self.text_input_area.setFont(font_input)

        font_info = QFont()
        font_info.setPointSize(DEFAULT_FONT_SIZE - 2)
        self.lbl_audio_file.setFont(font_info)


        self.update_icons() # Icons might need re-evaluation based on theme contrast
        current_pos_ms = self.progress_slider.value() if self.audio_duration_ms > 0 else 0
        self.update_display(forced_time_ms=current_pos_ms) # Refresh display with new theme colors

    def update_icons(self):
        self.setWindowIcon(QIcon.fromTheme("multimedia-player", self.style().standardIcon(QStyle.SP_MediaPlay)))
        # No btn_load_audio
        self.btn_stop.setIcon(QIcon.fromTheme("media-playback-stop", self.style().standardIcon(QStyle.SP_MediaStop)))
        self.lbl_volume_icon.setPixmap(QIcon.fromTheme("audio-volume-medium", self.style().standardIcon(QStyle.SP_MediaVolume)).pixmap(24, 24))
        # Play/Pause icon is updated in update_button_states
        self.update_button_states() # This will set initial play/pause icon

    def disable_audio_controls(self):
        # No btn_load_audio
        self.btn_play_pause.setEnabled(False)
        self.btn_stop.setEnabled(False)
        self.progress_slider.setEnabled(False)
        self.volume_slider.setEnabled(False)
        self.text_input_area.setEnabled(False)
        self.lbl_audio_file.setText("Audio: Unavailable (Pygame Mixer Error)")
        self.status_bar.showMessage("Audio system error. Playback disabled.")

    def apply_stylesheet(self):
        colors = self.theme_colors
        # Using more specific selectors for some items
        self.setStyleSheet(f"""
            QMainWindow, QWidget {{
                background-color: {colors['app_bg']};
                color: {colors['app_fg']};
                font-size: {DEFAULT_FONT_SIZE}px;
            }}
            QPushButton {{
                background-color: {colors['btn_bg']};
                border: 1px solid {colors['btn_border']};
                padding: 8px 15px;
                border-radius: 4px;
                min-width: 80px; /* For generic buttons if any were added */
                color: {colors['app_fg']};
            }}
            QPushButton#PlayPauseButton, QPushButton#StopButton {{ /* If you set objectNames */
                min-width: 40px; /* Override for small icon buttons */
            }}
            QPushButton:hover {{ background-color: {colors['btn_hover']}; }}
            QPushButton:pressed {{ background-color: {colors['btn_pressed']}; }}
            QPushButton:disabled {{
                background-color: {colors['btn_disabled_bg']};
                color: {colors['btn_disabled_fg']};
                border-color: {colors['btn_disabled_bg']};
            }}
            QLabel {{ color: {colors['label_fg']}; }}
            QLabel#AudioInfoLabel {{ color: {colors['label_info_fg']}; }}
            QLabel#SubtitleDisplayLabel {{
                padding: 20px;
                border: 2px solid {colors['subtitle_display_border']};
                background-color: {colors['subtitle_display_bg']};
                color: {colors['subtitle_display_fg']};
                border-radius: 8px;
            }}
            QTextEdit#TextInputArea {{
                background-color: {colors['text_edit_bg']};
                color: {colors['text_edit_fg']};
                border: 2px solid {colors['text_edit_border']};
                border-radius: 6px;
                padding: 10px;
            }}
            QTextEdit#TextInputArea::placeholder {{
                color: {colors['text_edit_placeholder']};
            }}
            QSlider::groove:horizontal {{
                border: 1px solid {colors['slider_groove_border']};
                height: 8px;
                background: {colors['slider_groove_bg']};
                margin: 2px 0;
                border-radius: 4px;
            }}
            QSlider::handle:horizontal {{
                background: {colors['slider_handle_bg']};
                border: 1px solid {colors['slider_handle_border']};
                width: 16px; margin: -4px 0; border-radius: 8px;
            }}
            QSlider::sub-page:horizontal {{
                background: {colors['slider_sub_page_bg']}; border-radius: 4px;
            }}
            QStatusBar {{
                background-color: {colors['statusbar_bg']}; color: {colors['statusbar_fg']};
            }}
            QMenuBar {{
                background-color: {colors['menu_bar_bg']}; color: {colors['menu_bar_fg']};
            }}
            QMenuBar::item {{
                background-color: transparent; padding: 4px 10px; margin: 2px; border-radius: 3px;
            }}
            QMenuBar::item:selected {{ background-color: {colors['menu_item_selected_bg']}; }}
            QMenu {{
                background-color: {colors['menu_bg']}; color: {colors['menu_fg']};
                border: 1px solid {colors['menu_border']}; padding: 4px;
            }}
            QMenu::item {{ padding: 6px 25px 6px 25px; border-radius: 3px; }}
            QMenu::item:selected {{ background-color: {colors['menu_item_selected_bg']}; }}
            QMenu::separator {{ height: 1px; background-color: {colors['menu_border']}; margin: 5px 0; }}
        """)

    def load_audio_dialog(self):
        if self.pygame_mixer_failed:
            self.status_bar.showMessage("Audio system error. Cannot load audio.", 5000)
            self.reset_playback_state(reset_text_input=False) # Keep text, show error
            return

        # self.current_typed_text is guaranteed to be set by handle_text_entry_and_trigger_audio_load
        if not self.current_typed_text: # Should not happen with new flow
            QMessageBox.critical(self, "Internal Error", "No text found to associate with audio.")
            self.reset_playback_state()
            return

        path, _ = QFileDialog.getOpenFileName(self, "Select Audio File", os.path.expanduser("~"), "Audio Files (*.mp3 *.wav *.ogg)")
        if path:
            self.audio_path = path
            try:
                if pygame.mixer.music.get_busy():
                    pygame.mixer.music.stop()
                pygame.mixer.music.load(self.audio_path)
                temp_sound = pygame.mixer.Sound(self.audio_path)
                self.audio_duration_ms = int(temp_sound.get_length() * 1000)
                del temp_sound

                if self.audio_duration_ms <= 0:
                    raise pygame.error("Audio file has zero or negative duration.")

                self.lbl_audio_file.setText(f"Audio: {os.path.basename(path)}")
                self.lbl_total_time.setText(self.format_time(self.audio_duration_ms))
                self.progress_slider.setMaximum(self.audio_duration_ms)

                # Create the single subtitle item
                sub = pysrt.SubRipItem()
                sub.text = self.current_typed_text
                sub.start = pysrt.SubRipTime(milliseconds=0)
                sub.end = pysrt.SubRipTime(milliseconds=self.audio_duration_ms)
                sub.start_ms = 0
                sub.end_ms = self.audio_duration_ms
                self.subs = [sub]

                self.subtitle_display_label.setText(html.escape(self.current_typed_text)) # Show plain text, ready for play
                self.status_bar.showMessage("Text and audio loaded. Ready to play.", 5000)

            except pygame.error as e:
                self.lbl_audio_file.setText("Audio: Error loading or invalid file!")
                QMessageBox.warning(self, "Audio Load Error", f"Could not load or use audio file: {e}")
                self.audio_path = None
                self.audio_duration_ms = 0
                self.subs = None
                # Keep current_typed_text, allow user to try loading audio again
                self.reset_playback_state(reset_text_input=False) # Resets audio state but keeps text
                self.subtitle_display_label.setText(html.escape(self.current_typed_text) + "\n\n(Audio load failed. Press Enter on text to try again.)")
                return # Important to return after handling error
            finally:
                self.update_button_states() # Always update states
        else: # User cancelled file dialog
            self.status_bar.showMessage("Audio selection cancelled. Press Enter on your text to try again.", 5000)
            self.audio_path = None # Ensure it's None
            self.subs = None
            self.lbl_audio_file.setText("Audio: Not loaded (Selection cancelled)")
            if self.current_typed_text:
                self.subtitle_display_label.setText(html.escape(self.current_typed_text) + "\n\n(Audio selection cancelled. Press Enter on text again.)")
            else: # Should not happen, but as a fallback
                self.subtitle_display_label.setText("Type text, press Enter, then select audio.")
            self.update_button_states()

    def update_button_states(self):
        if self.pygame_mixer_failed:
            self.disable_audio_controls()
            return

        can_operate = bool(self.subs and self.audio_path and self.audio_duration_ms > 0)
        self.btn_play_pause.setEnabled(can_operate)
        self.progress_slider.setEnabled(can_operate)
        is_playing_or_paused = pygame.mixer.music.get_busy() or self.is_paused
        self.btn_stop.setEnabled(can_operate and is_playing_or_paused)

        if pygame.mixer.music.get_busy() and not self.is_paused:
            self.btn_play_pause.setIcon(QIcon.fromTheme("media-playback-pause", self.style().standardIcon(QStyle.SP_MediaPause)))
            self.btn_play_pause.setToolTip("Pause (Spacebar)")
        else:
            self.btn_play_pause.setIcon(QIcon.fromTheme("media-playback-start", self.style().standardIcon(QStyle.SP_MediaPlay)))
            self.btn_play_pause.setToolTip("Play (Spacebar)")

    def toggle_play_pause(self):
        if self.pygame_mixer_failed or not self.subs or not self.audio_path: return
        # ... (rest of toggle_play_pause is fine)
        if pygame.mixer.music.get_busy() and not self.is_paused: # Playing -> Pause
            pygame.mixer.music.pause()
            self.is_paused = True
            self.timer.stop()
            self.status_bar.showMessage("Paused.", 3000)
        else: # Paused or Stopped -> Play/Resume
            if self.is_paused: # Resuming
                pygame.mixer.music.unpause()
                self.status_bar.showMessage("Resumed.", 3000)
            else: # Starting from beginning or seeked position
                self.playback_start_offset_ms = self.progress_slider.value()
                try:
                    # Ensure music is loaded if it was stopped/unloaded due to error or reset
                    if not pygame.mixer.music.get_busy() and self.audio_path:
                         pygame.mixer.music.load(self.audio_path)

                    pygame.mixer.music.play(loops=0, start=self.playback_start_offset_ms / 1000.0)
                except pygame.error as e:
                    QMessageBox.warning(self, "Playback Error", f"Could not start playback: {e}")
                    self.reset_playback_state(reset_text_input=False)
                    return
                self.status_bar.showMessage("Playing...", 3000)
            self.is_paused = False
            self.timer.start()
        self.update_button_states()


    def stop_playback(self):
        if self.pygame_mixer_failed: return
        # ... (rest of stop_playback is fine)
        if pygame.mixer.music.get_busy() or self.is_paused:
            pygame.mixer.music.stop()
            self.timer.stop()
            if self.subs and self.subs[0]:
                 self.subtitle_display_label.setText(html.escape(self.subs[0].text_without_tags))
            elif self.current_typed_text: # Fallback if subs not fully formed but text exists
                 self.subtitle_display_label.setText(html.escape(self.current_typed_text))
            else:
                 self.subtitle_display_label.setText("Playback stopped.")

            self.current_sub_index = -1
            self.is_paused = False
            self.progress_slider.setValue(0)
            self.lbl_current_time.setText(self.format_time(0))
            self.playback_start_offset_ms = 0
            self.status_bar.showMessage("Stopped.", 3000)
        self.update_button_states()


    def reset_playback_state(self, reset_text_input: bool = True, from_menu: bool = False):
        self.stop_playback() # This also stops the timer and updates buttons partially

        self.progress_slider.setValue(0)
        self.lbl_current_time.setText(self.format_time(0))
        self.playback_start_offset_ms = 0
        self.current_sub_index = -1

        if reset_text_input:
            self.text_input_area.clear()
            self.current_typed_text = ""
            self.subs = None
            self.audio_path = None
            self.audio_duration_ms = 0
            self.lbl_audio_file.setText("Audio: Not loaded. Type text and press Enter.")
            self.lbl_total_time.setText(self.format_time(0))
            self.progress_slider.setMaximum(100)
            self.progress_slider.setValue(0)
            self.subtitle_display_label.setText("Type text above, press Enter, then select audio.")
            self.status_bar.showMessage("Ready. Type text and press Enter." if from_menu else "State reset.")
            if not self.pygame_mixer_failed: self.text_input_area.setFocus()
        else: # Keep text, reset audio-related things
            self.subs = None # If audio failed, subs are invalid
            # self.audio_path = None # Keep audio_path if it was valid but failed to play? No, clear it.
            # self.audio_duration_ms = 0
            if not self.audio_path: # If audio path was never set or cleared due to error
                self.lbl_audio_file.setText(f"Audio: Not loaded. Press Enter on text to select.")
            self.lbl_total_time.setText(self.format_time(0))
            self.progress_slider.setMaximum(100)
            self.progress_slider.setValue(0)
            if self.current_typed_text:
                self.subtitle_display_label.setText(html.escape(self.current_typed_text) + "\n\n(Audio not loaded/reset. Press Enter on text.)")
            else: # Should not be reached if reset_text_input is False and current_typed_text is empty
                self.subtitle_display_label.setText("Type text, press Enter, then select audio.")

        self.update_button_states()


    def format_time(self, ms: int) -> str:
        # ... (format_time is fine)
        s = max(0, ms // 1000)
        m, s = divmod(s, 60)
        h, m = divmod(m, 60)
        return f"{h:02d}:{m:02d}:{s:02d}"

    def update_display(self, forced_time_ms: Optional[int] = None):
        # ... (update_display logic for highlighting is largely fine)
        if self.pygame_mixer_failed: return

        actual_current_song_time_ms: int
        playback_active = pygame.mixer.music.get_busy() or self.is_paused

        if forced_time_ms is not None:
            actual_current_song_time_ms = forced_time_ms
        elif playback_active:
            time_elapsed_since_play_call = pygame.mixer.music.get_pos()
            if time_elapsed_since_play_call == -1:
                # If get_pos() is -1, music likely stopped or hasn't started correctly.
                # Check if it should have finished.
                if self.audio_duration_ms > 0 and not self.is_paused and self.timer.isActive() and \
                   (self.playback_start_offset_ms + UPDATE_INTERVAL_MS * 2) >= self.audio_duration_ms : # A bit of leeway
                    self.stop_playback()
                    self.status_bar.showMessage("Playback finished.", 3000)
                # else:
                    # It might be an error or just stopped before end. stop_playback handles button states.
                    # If timer is active and not paused, means it was playing and unexpectedly stopped.
                    # if self.timer.isActive() and not self.is_paused: self.stop_playback()
                return

            actual_current_song_time_ms = self.playback_start_offset_ms + time_elapsed_since_play_call

            if not self.user_is_seeking and pygame.mixer.music.get_busy() and not self.is_paused:
                if self.audio_duration_ms > 0:
                    display_slider_time = min(actual_current_song_time_ms, self.audio_duration_ms)
                    self.progress_slider.setValue(display_slider_time)
        else: # Not playing and not paused
            if self.timer.isActive(): # Timer still active means it just finished
                self.stop_playback()
                self.status_bar.showMessage("Playback naturally finished.", 3000)
            return

        if self.audio_duration_ms > 0:
            actual_current_song_time_ms = min(actual_current_song_time_ms, self.audio_duration_ms)
        actual_current_song_time_ms = max(0, actual_current_song_time_ms)

        self.lbl_current_time.setText(self.format_time(actual_current_song_time_ms))

        if not self.subs: return

        active_sub = self.subs[0]
        if active_sub.start_ms <= actual_current_song_time_ms < active_sub.end_ms:
            if self.current_sub_index != 0: self.current_sub_index = 0

            words = active_sub.text_without_tags.split()
            num_words = len(words)
            if num_words == 0:
                self.subtitle_display_label.setText(html.escape(active_sub.text_without_tags))
                return

            time_into_sub = actual_current_song_time_ms - active_sub.start_ms
            sub_duration = active_sub.end_ms - active_sub.start_ms
            highlight_idx = -1

            if sub_duration > 0:
                progress_ratio = time_into_sub / sub_duration
                highlight_idx = int(progress_ratio * num_words)
                highlight_idx = min(highlight_idx, num_words - 1)

            display_text_parts = []
            highlight_color = self.theme_colors['subtitle_highlight_word']
            font_size_px = SUBTITLE_DISPLAY_FONT_SIZE

            for i, word in enumerate(words):
                escaped_word = html.escape(word)
                style = f"font-size: {font_size_px}px;"
                if i == highlight_idx:
                    style += f" color: {highlight_color}; font-weight: bold;"
                display_text_parts.append(f"<span style='{style}'>{escaped_word}</span>")
            self.subtitle_display_label.setText(" ".join(display_text_parts))
        elif self.current_sub_index != -1: # Time is outside sub
            self.subtitle_display_label.setText(html.escape(self.subs[0].text_without_tags)) # Show plain text
            self.current_sub_index = -1

        self.update_button_states() # Keep buttons in sync with playback state


    def slider_pressed(self):
        # ... (slider_pressed is fine)
        if self.pygame_mixer_failed: return
        self.user_is_seeking = True
        if pygame.mixer.music.get_busy() and not self.is_paused:
            pygame.mixer.music.pause()

    def slider_released(self):
        # ... (slider_released is fine)
        if self.pygame_mixer_failed: return
        self.user_is_seeking = False
        if not self.audio_path or not self.subs: # Ensure we have audio and subs to play
            self.progress_slider.setValue(self.playback_start_offset_ms) # Revert slider if no audio
            return

        seek_ms = self.progress_slider.value()
        self.playback_start_offset_ms = seek_ms

        try:
            # Ensure music is loaded before playing, especially if it was stopped.
            if not pygame.mixer.music.get_busy() and self.audio_path:
                pygame.mixer.music.load(self.audio_path)

            pygame.mixer.music.play(loops=0, start=seek_ms / 1000.0)
            if self.is_paused:
                pygame.mixer.music.pause()
            else:
                self.timer.start()
            self.status_bar.showMessage(f"Seeked to {self.format_time(seek_ms)}", 3000)
        except pygame.error as e:
            QMessageBox.warning(self, "Seek Error", f"Could not seek audio: {e}")
            self.reset_playback_state(reset_text_input=False)
        self.update_display(forced_time_ms=seek_ms)
        self.update_button_states()

    def seek_audio_slider_moved(self, value_ms: int):
        # ... (seek_audio_slider_moved is fine)
        if self.pygame_mixer_failed: return
        self.lbl_current_time.setText(self.format_time(value_ms))
        if self.user_is_seeking and self.subs: # Live update while dragging
             self.update_display(forced_time_ms=value_ms)


    def set_volume(self, value: int):
        # ... (set_volume is fine)
        if self.pygame_mixer_failed: return
        pygame.mixer.music.set_volume(value / 100.0)
        self.status_bar.showMessage(f"Volume: {value}%", 2000)


    def closeEvent(self, event: QEvent):
        self.stop_playback()
        if not self.pygame_mixer_failed:
            pygame.mixer.music.stop()
            pygame.mixer.quit()
        print("Application closed.")
        event.accept()

if __name__ == "__main__":
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)

    if sys.platform == "win32":
        import ctypes
        myappid = u'mycompany.myproduct.textaudiosyncpro.1.1' # Updated version
        try:
            ctypes.windll.shell32.SetCurrentProcessExplicitAppUserModelID(myappid)
        except AttributeError:
            print("Warning: Could not set AppUserModelID.")

    window = TextAudioSynchronizerPro()
    window.show()
    sys.exit(app.exec())

pygame 2.6.1 (SDL 2.28.4, Python 3.9.21)
Hello from the pygame community. https://www.pygame.org/contribute.html
Application closed.


error: mixer not initialized

SystemExit: 0

c:\Users\ANAS\anaconda3\envs\ML\lib\site-packages\IPython\core\interactiveshell.py:3513: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
